# Playground Series S6E8 — Predicting Smartphone Addiction
## 📱 Robust Ensembling & Target Encoding Pipeline（解説付き学習用コピー）

- **コンペ**: [Predicting Smartphone Addiction（Playground Series S6E8）](https://www.kaggle.com/competitions/playground-series-s6e8)（2,442チーム・残り11日）
- **原著者**: Koushik Kumar Dinda（[@koushikkumardinda](https://www.kaggle.com/koushikkumardinda)）
- **元notebook**: https://www.kaggle.com/code/koushikkumardinda/robust-ensembling-target-encoding-pipeline
- **スコア**: Public 0.96464・44 votes（Bronze）・6 comments
- **レビュー日**: 2026-08-21

> ⚠️ **これは学習目的の解説付き写しです。** 原著のコードは変更していませんが、出力は含んでおらず未実行です。

---

### なぜ「最高スコア」ではなく、このnotebookを選んだか

このコンペは終盤に入り、公開LBの上位帯（0.9709〜0.9712）を占めるnotebookの多くが
**他人の `submission.csv` を読み込んでランク平均するだけ**の内容になっています。
実際、今日の選定候補だった Public 0.97099 の「TOP-1」notebook は本体がわずか5セルで、
中身は3つの公開提出をランク平均する4行でした。スコアは高くても、**学べる手法がほぼありません**。

本notebookは Public 0.96464 と上位帯には届きませんが、
**データの読み込みからターゲットエンコーディング、10-fold CV、ランク平均、メタモデルまでが1本につながった
完全なパイプライン**です。しかも後述するとおり、**「なぜ伸びなかったのか」を追える具体的なバグと設計の綻び**
まで含んでおり、学習素材としてはこちらのほうが遥かに価値があります。
（詳細は同ディレクトリの `README.md` の「改善点の考察」を参照。）

---

### 評価指標について

**タスク**: 表形式の特徴量（睡眠時間、1日のスクリーンタイム、通知数、年齢、ストレスレベル等）から、
そのユーザーがスマートフォン依存（`addicted_label` = 1）かどうかの**確率**を予測する二値分類です。

**指標**: **ROC-AUC**（ROC曲線下面積）。
全ての「陽性1件・陰性1件」のペアについて、陽性のほうに高いスコアを付けられた割合に等しく、
**予測値の順位だけで決まり、絶対値には依存しません**。0.5がランダム、1.0が完璧です。

**なぜこの指標か**: AUCは**しきい値を1つに決めなくてよい**指標です。
「依存と判定する境界をどこに引くか」は用途（介入コスト、見逃しの重大さ）によって変わるので、
コンペ側でしきい値を固定してしまうと、モデルの良し悪しではなく
**しきい値選びの巧拙**を競うことになってしまいます。
また、クラス比が偏っていても Accuracy のように「多数派に全部倒す」で高得点を取ることができません。

**本notebookの手法が指標にどう向き合っているか**:
AUCが**順位だけの指標**であることを強く意識した設計になっています。
LightGBM と XGBoost の出力確率を、そのまま平均せず
**`rank(pct=True)` でパーセンタイル（順位）に変換してから**混ぜているのがそれです。
2つのモデルは同じ「上位1%」を表すのに 0.4 と 0.6 という違う確率を出すかもしれませんが、
順位に直せば**どちらも 0.99** になります。AUCで評価されるなら、
スケールを揃えるのに**キャリブレーションではなく順位化で十分**、という判断です。


### 📱 Kaggle S6E8: Smartphone Addiction Prediction

The goal of this competition is to predict the probability of a user having a smartphone addiction (`addicted_label`: 0 = Not Addicted, 1 = Addicted) evaluated on the Area under the ROC curve (ROC AUC) metric[cite: 1]. 

Because this dataset is synthetically generated from a much smaller real-world dataset of around 7,500 rows, it contains certain generator artifacts that tree-based models struggle with[cite: 1]. This notebook splits the provided script to apply the top strategies for this competition: **Stringified Target Encoding**, **Out-of-Fold (OOF) Stacking**, and **Rank Averaging**[cite: 1].

#### 1. Environment Initialization
First, we load the standard Kaggle environment libraries to handle data processing and ensure our input paths are correct.

### 実行環境の確認（Kaggleの定型セル）

**何をしているか**: `numpy` と `pandas` を読み込み、`/kaggle/input` 以下を歩いて
添付されている全ファイルのパスを印字しています。

**なぜそうするのか**: Kaggleが新規notebookに自動挿入する定型セルです。
実質的な意味は**「データがどこにマウントされたかを目で確認する」**こと。
競技データ、他人のデータセット、モデル重みなどのパスは環境によって変わるため、
最初に一覧を出しておくと後のパス指定でつまずきません。

初心者向けの補足として、Kaggleの実行環境には2つの重要なディレクトリがあります。

- `/kaggle/input/` … **読み取り専用**。競技データや添付データセットが置かれる
- `/kaggle/working/` … 書き込み可能（20GBまで）。`submission.csv` はここに出力する

なお、この定型セルはスコアには一切影響しません。

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

#### 2. Configuration & The "Stringify" Trick
Tree-based models have a hard time finding logical, continuous splits on grid-like synthetic data[cite: 1]. To solve this, a top tactic is to convert continuous numerical features (such as `sleep_hours`, `notifications_per_day`, and `daily_screen_time_hours`) into categorical strings using `.astype(str)`[cite: 1]. Later, we will apply Target Encoding to these strings to create a lookup table that reverse-engineers the synthetic generator's logic[cite: 1].

### 設定と「文字列化（Stringify）」トリック

**何をしているか**: 2つのことをしています。

1. ライブラリの読み込みと設定（10-fold、シード42、対象列 `addicted_label`）
2. **連続値の列を `.astype(str)` で文字列に変換する**
   （`sleep_hours`, `notifications_per_day`, `daily_screen_time_hours`）

そのあと `select_dtypes(include=['object','category'])` で
**文字列型になっている列を全部**拾い、ターゲットエンコーディングの対象にしています。

**なぜそうするのか（ここが本notebookの肝）**:

Playground Series のデータは、**小さな実データ（このコンペでは約7,500行）から
生成モデルで水増しした合成データ**です。合成データには独特の癖があり、
連続値のはずの列が**実際にはごく少数の離散値しか取らない**——つまり
**値が格子状に並ぶ**ことがよくあります。

木モデル（LightGBM/XGBoost）は「x < 4.5 なら左」という**大小関係の分割**しか作れません。
値が格子状で、しかも各格子点ごとに目的変数の平均がバラバラに決まっているような場合、
**大小関係には意味がなく**、木は多数の分割を費やしても効率よく表現できません。

そこで、連続値を**カテゴリ（文字列）として扱い直し**、
次のセルでターゲットエンコーディングをかけます。すると各格子点が
**「その値のときの陽性率」というルックアップ表**に変換されます。
これは実質的に、**合成データの生成ルールを逆算している**のと同じことです。

> ⚠️ 注意: これは「合成データだから効く」トリックです。
> 本物の連続値（測定値）に同じことをすると、**未知の値が丸ごと未知カテゴリになる**ため、
> 汎化性能はむしろ落ちます。手法とデータの相性を見極める練習台として良い例です。

> 用語: **ターゲットエンコーディング** = カテゴリの各水準を、
> その水準における目的変数の平均値（ここでは陽性率）で置き換える手法。

In [ ]:
import pandas as pd
import numpy as np
import warnings
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from category_encoders import TargetEncoder

from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier

# Keep the console output clean
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =========================================================
# 1. Configuration & Data Loading
# =========================================================
print("Loading data...")
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
TEST_PATH = '/kaggle/input/competitions/playground-series-s6e8/test.csv'
TARGET = 'addicted_label'
N_FOLDS = 10
RANDOM_SEED = 42

df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

# =========================================================
# 2. Feature Engineering: The "Stringify" Trick
# =========================================================
# In synthetic tabular data, converting continuous features with low 
# unique value counts into categorical strings helps tree models map 
# the generator's artifacts via Target Encoding.

features_to_stringify = ['sleep_hours', 'notifications_per_day', 'daily_screen_time_hours']

for col in features_to_stringify:
    df_train[col] = df_train[col].astype(str)
    df_test[col] = df_test[col].astype(str)

# Automatically grab ALL string/object columns to pass to the TargetEncoder.
# This captures the 3 we just stringified PLUS 'gender', 'stress_level', etc.
features_to_encode = df_train.select_dtypes(include=['object', 'category']).columns.tolist()

X = df_train.drop(columns=[TARGET, 'id'])
y = df_train[TARGET]
X_test = df_test.drop(columns=['id'])

#### 3. Strict 10-Fold Cross-Validation Setup
When using Target Encoding, the most significant risk is data leakage, which occurs if you encode features using the target variable of the entire dataset before splitting[cite: 1]. This results in artificially inflated validation scores[cite: 1]. 

To execute this correctly, the encoder mapping must be fit **only** on the training fold, and then used to transform the validation fold and test set[cite: 1]. We will do this inside a 10-Fold Stratified CV loop using both LightGBM and XGBoost base models[cite: 1].

### 層化10-fold CV：ターゲットエンコーディングを「fold内でだけ」学習する

**何をしているか**: `StratifiedKFold(n_splits=10)` のループの中で、fold ごとに

1. `TargetEncoder(smoothing=10)` を **学習用foldだけで `fit`**
2. その encoder で検証fold・テストデータを `transform`
3. LightGBM と XGBoost を早期終了つきで学習
4. OOF予測とテスト予測（`/N_FOLDS` で平均）を蓄積

を繰り返します。

**なぜそうするのか（最重要ポイント）**:

ターゲットエンコーディングは、**目的変数の情報を特徴量に埋め込む**手法です。
分割前に全データでエンコードすると、**検証foldの正解が特徴量経由で漏れ込みます**（リーク）。
その結果、CVスコアだけが不自然に高く出て、LBでは全く再現しません。
本notebookのように**`fit` を学習foldに限定する**のが唯一の正しいやり方です。

補足しておきたい細かい設計:

- **`smoothing=10`**: 出現回数が少ないカテゴリの平均は当てになりません。
  平滑化は、そのカテゴリの平均を**全体平均のほうへ引き戻す**強さを決めます。
  値を大きくすると保守的（全体平均寄り）、小さくすると攻撃的になります。
- **`StratifiedKFold`（層化）**: 各foldでクラス比を揃えます。
  不均衡データで普通の `KFold` を使うと、fold ごとに陽性率が変動し、AUCの見積もりが不安定になります。
- **早期終了（early stopping）**: `n_estimators=1500` と多めに設定しつつ、
  検証スコアが50回改善しなければ打ち切ります。木の本数を手で決めなくてよくなる定番の作法です。
- **テスト予測の作り方**: `test_lgbm += predict_proba(...) / N_FOLDS`。
  10個のfoldモデルの平均を取っており、これ自体が**10モデルのアンサンブル**として機能します。

> ⚠️ 1点だけ注意: 早期終了の検証セットに、**そのまま検証fold（`X_va`）を使っています**。
> 木の本数を検証foldで選び、同じfoldでOOFスコアを測るので、
> **OOFスコアはごくわずかに楽観的**になります。厳密にやるなら内側にもう一段CVを切ります。

In [ ]:
# =========================================================
# 3. Stratified K-Fold Training Loop
# =========================================================
print(f"Starting {N_FOLDS}-Fold Cross Validation...")
kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# Arrays to hold our Out-Of-Fold (OOF) and Test predictions
oof_lgbm = np.zeros(len(df_train))
oof_xgb = np.zeros(len(df_train))

test_lgbm = np.zeros(len(df_test))
test_xgb = np.zeros(len(df_test))

cv_scores_lgbm = []
cv_scores_xgb = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n--- Fold {fold + 1} ---")
    
    # Safely split data
    X_tr, X_va = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]
    
    # -----------------------------------------------------
    # A. In-Fold Target Encoding (Prevents Leakage)
    # -----------------------------------------------------
    encoder = TargetEncoder(cols=features_to_encode, smoothing=10)
    X_tr = encoder.fit_transform(X_tr, y_tr)
    X_va = encoder.transform(X_va)
    X_te_fold = encoder.transform(X_test)
    
    # -----------------------------------------------------
    # B. Train LightGBM
    # -----------------------------------------------------
    lgb_model = LGBMClassifier(
        n_estimators=1500, 
        learning_rate=0.03, 
        random_state=RANDOM_SEED, 
        verbose=-1
    )
    lgb_model.fit(
        X_tr, y_tr, 
        eval_set=[(X_va, y_va)], 
        callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    oof_lgbm[val_idx] = lgb_model.predict_proba(X_va)[:, 1]
    test_lgbm += lgb_model.predict_proba(X_te_fold)[:, 1] / N_FOLDS
    cv_scores_lgbm.append(roc_auc_score(y_va, oof_lgbm[val_idx]))
    
    # -----------------------------------------------------
    # C. Train XGBoost
    # -----------------------------------------------------
    xgb_model = XGBClassifier(
        n_estimators=1500, 
        learning_rate=0.03, 
        random_state=RANDOM_SEED, 
        early_stopping_rounds=50, 
        eval_metric='auc',
        verbosity=0
    )
    xgb_model.fit(
        X_tr, y_tr, 
        eval_set=[(X_va, y_va)], 
        verbose=False
    )
    
    oof_xgb[val_idx] = xgb_model.predict_proba(X_va)[:, 1]
    test_xgb += xgb_model.predict_proba(X_te_fold)[:, 1] / N_FOLDS
    cv_scores_xgb.append(roc_auc_score(y_va, oof_xgb[val_idx]))

print("\n=========================================================")
print(f"LightGBM Mean CV: {np.mean(cv_scores_lgbm):.5f}")
print(f"XGBoost Mean CV:  {np.mean(cv_scores_xgb):.5f}")
print("=========================================================")

#### 4. Rank Averaging and Optuna Optimization
Before combining out-of-fold (OOF) predictions, it is heavily recommended to rank-average them[cite: 1]. Models output probabilities on different scales (e.g., one model may output 0.4 and another 0.6 to represent the same top percentile of risk)[cite: 1]. Converting these probabilities to percentiles (ranks) neutralizes absolute scaling differences, placing all models on the exact same uniform distribution[cite: 1].

Furthermore, using Optuna to constrain the blend weights to positive fractions that sum to 1.0 is highly robust and often safer than training a new Meta-Model, which may overfit[cite: 1].

### ランク変換 → メタ特徴量 → 浅いメタモデルによるスタッキング

**何をしているか**: 4段階です。

1. LightGBM と XGBoost の OOF予測・テスト予測を `rank(pct=True)` で**パーセンタイル**に変換
2. それを列に持つメタ用のDataFrameを作り、生の特徴量2つ（`age`, `daily_screen_time_hours`）を追加
3. **極端に正則化した LightGBM**（`max_depth=2`, `num_leaves=3`, `min_child_samples=100`,
   `learning_rate=0.01`, `colsample_bytree=0.5`）をメタモデルとして学習
4. `cross_val_score` でメタモデルのCV AUCを確認し、全データで再学習して提出を作成

**なぜそうするのか**:

- **ランク変換の理由**: AUCは順位のみで決まるので、モデル間の確率スケールの違いを
  順位化で潰してしまうのが最も安全です。「片方のモデルが自信過剰で平均を支配する」事故を防げます。
- **メタモデルを極端に浅くする理由**: メタモデルの入力にはOOF予測が含まれます。
  自由度を与えすぎると、メタモデルは**OOF予測をそのまま暗記**してしまい、
  テストでは再現しない関係を学びます。深さ2・葉3枚は事実上「決定株（decision stump）」で、
  **「OOF予測を微調整する」以上のことをさせない**ための縛りです。
- **生の特徴量を2つだけ足す理由**: 「この年齢帯・このスクリーンタイム帯では
  ベースモデルが系統的に外している」といった**条件つきの補正**を学ばせる狙いです。
  ただし入れすぎるとメタモデルが第2のベースモデルになってしまうので2つに絞っています。

> ⚠️ **このセルには実際にバグがあります。**
> `meta_X_train` の列名は `lgbm_oof_rank` / `xgb_oof_rank`、
> `meta_X_test` の列名は `lgbm_test_rank` / `xgb_test_rank` で、**名前が一致していません**。
> LightGBM は特徴量名を保持するため、この不一致は警告か例外を招き、
> 少なくとも**「学習時と推論時で同じ意味の列が同じ名前で渡っていない」**状態です。
> このnotebookのスコア 0.96464 が、素の LightGBM 単体（0.96965）にすら届いていない
> 一因はここにある可能性が高い。詳しくは `README.md` の改善点を参照してください。
>
> **教訓**: スタッキングでは、学習側とテスト側のメタ特徴量を
> **同じ関数で作る**か、少なくとも `assert list(train.columns) == list(test.columns)` を
> 1行入れておくべきです。

In [ ]:
# =========================================================
# 4. Rank Transformation & Meta-Feature Preparation
# =========================================================
print("\nPreparing Meta-Features...")

# Convert raw probabilities to percentiles (0.0 to 1.0) so both 
# models are evaluated on the exact same uniform distribution scale.
rank_oof_lgbm = pd.Series(oof_lgbm).rank(pct=True).values
rank_oof_xgb = pd.Series(oof_xgb).rank(pct=True).values
rank_test_lgbm = pd.Series(test_lgbm).rank(pct=True).values
rank_test_xgb = pd.Series(test_xgb).rank(pct=True).values

# Initialize Meta-Train and Meta-Test DataFrames with OOF ranks
meta_X_train = pd.DataFrame({
    'lgbm_oof_rank': rank_oof_lgbm,
    'xgb_oof_rank': rank_oof_xgb
})

meta_X_test = pd.DataFrame({
    'lgbm_test_rank': rank_test_lgbm,
    'xgb_test_rank': rank_test_xgb
})

# Inject original raw features for context (loading fresh to avoid the stringified versions from earlier)
raw_train = pd.read_csv(TRAIN_PATH)
raw_test = pd.read_csv(TEST_PATH)

# Select 2 highly important features to pass to the Meta-Model
raw_features_to_add = ['age', 'daily_screen_time_hours']
for col in raw_features_to_add:
    if col in raw_train.columns:
        meta_X_train[col] = raw_train[col]
        meta_X_test[col] = raw_test[col]

# =========================================================
# 5. Train Shallow LightGBM Meta-Model
# =========================================================
print("Training Shallow LightGBM Meta-Model...")

# We use extreme regularization to prevent the Meta-Model from 
# memorizing the OOF probabilities and force it to use raw features.
meta_model = LGBMClassifier(
    max_depth=2,                 # Restrict to decision stumps
    num_leaves=3,                # Max 3 leaves per tree
    min_child_samples=100,       # Prevent fitting to micro-clusters
    learning_rate=0.01,          # Slow and steady learning
    n_estimators=300,
    colsample_bytree=0.5,        # Force feature subsampling
    subsample=0.8,               # Row subsampling
    subsample_freq=1,
    random_state=RANDOM_SEED,
    verbose=-1
)

# Cross-validate the Meta-Model to verify score improvement
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(
    meta_model, 
    meta_X_train, 
    y, 
    cv=10, 
    scoring='roc_auc'
)
print(f"Stacked Meta-Model CV AUC: {cv_scores.mean():.5f}")

# Fit on the full meta-training set
meta_model.fit(meta_X_train, y)

# =========================================================
# 6. Final Submission
# =========================================================
print("\nGenerating final stacked submission file...")
final_predictions = meta_model.predict_proba(meta_X_test)[:, 1]

submission = pd.DataFrame({
    'id': df_test['id'],
    TARGET: final_predictions
})

submission.to_csv('submission.csv', index=False)
print("Success! Saved to submission.csv")